# AML_04 — Comprendre et entraîner un réseau neuronal

**QUID :** comment les poids apprennent-ils, et le réseau mérite-t-il sa place face aux arbres ?

Même compétition, mêmes données officielles et mêmes folds que J2. Les scores sont locaux ; aucune soumission automatique.

Chaque numéro « cellule » désigne la position physique dans ce notebook, en comptant les textes. Exécuter dans l’ordre. Installation conservée depuis le ZIP initial ; aucun git pull.

Avant la séance : dans une cellule temporaire de Jupyter, exécuter `%pip install "torch>=2.4,<3"`, puis redémarrer le noyau. PyTorch utilise le CPU dans cet atelier ; aucun GPU requis.

[Guide de démarrage](https://github.com/MathieuDWeill/ai-course-student/blob/main/usecase/MyLittleMLLab/DEMARRER_J4.md) · [Cours rédigé](https://github.com/MathieuDWeill/ai-course-student/blob/main/usecase/MyLittleMLLab/lectures/AML_04_cours.md)


## Le contrat et PyTorch

Même protocole. Le notebook fonctionne sans J3 exécuté, mais comparer au tableau J3 si disponible.

Avant calcul : prévoir une sortie. Après calcul : nommer le résultat, sa limite et la décision.

In [ ]:
from pathlib import Path
import sys, json, time, hashlib, os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
HERE=Path.cwd()
ROOT=next((p if (p/'mllab/competition.py').exists() else p/'student') for p in [HERE,*HERE.parents] if (p/'mllab/competition.py').exists() or (p/'student/mllab/competition.py').exists())
sys.path.insert(0,str(ROOT))
from mllab.competition import subsets, make_model, xy, SPECS
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import roc_auc_score
from sklearn.base import clone
LIMIT=int(os.environ.get('MLLAB_LIMIT','15000'))
tr,va,test,sample,protocol_record=subsets('fraud',root=ROOT,limit=LIMIT)
X,y=xy(tr,'fraud'),tr.is_fraud.astype(int)
Xv,yv=xy(va,'fraud'),va.is_fraud.astype(int)
folds=list(StratifiedKFold(3,shuffle=True,random_state=2026).split(X,y))
assert not set(tr.id)&set(va.id)
RUN=ROOT/'runs/J4';RUN.mkdir(parents=True,exist_ok=True)
ids_sha=hashlib.sha256(tr.id.to_numpy().tobytes()).hexdigest()
print('Train :',len(tr),'fraudes :',int(y.sum()),'| contrôle local :',len(va))
print('3 folds fixes ; ROC-AUC ; CPU ; même protocole que J2.')
print('Le contrôle local a déjà été consulté en J2 : il ne constitue plus un test vierge.')

try:
 import torch
 from torch import nn
except ImportError as exc:
 raise ImportError('Installer dans une cellule temporaire : %pip install "torch>=2.4,<3", puis redémarrer le noyau.') from exc
torch.set_num_threads(2)
print('PyTorch',torch.__version__,'; appareil CPU explicite.')

## Un poids, un gradient, un pas

Exemple scalaire pédagogique ; sa perte quadratique sert à comprendre le mécanisme, pas à remplacer la perte de classification.

Avant calcul : prévoir une sortie. Après calcul : nommer le résultat, sa limite et la décision.

In [ ]:
w=torch.tensor(2.,requires_grad=True)
loss=(3*w-1)**2
loss.backward()
print('Avant step : poids =',w.item(),'perte =',loss.item(),'gradient =',w.grad.item())
assert w.item()==2 and w.grad.item()==30
for lr in [.01,1.]:
 new_w=2-lr*w.grad.item()
 print('Learning rate',lr,'→ poids',new_w,'→ perte',(3*new_w-1)**2)

## zero_grad, forward, backward, step

Backward calcule les gradients ; seul step modifie les paramètres dans cette boucle. zero_grad évite leur accumulation involontaire.

Avant calcul : prévoir une sortie. Après calcul : nommer le résultat, sa limite et la décision.

In [ ]:
w=nn.Parameter(torch.tensor(2.));optimizer=torch.optim.SGD([w],lr=.01)
optimizer.zero_grad();pred=3*w;loss=(pred-1)**2
loss.backward();before=w.detach().item();gradient=w.grad.item()
optimizer.step()
print('Avant :',before,'gradient :',gradient,'après :',w.item())
assert np.isclose(w.item(),1.7)
# Deux backward sans remise à zéro additionnent les gradients.
w2=torch.tensor(2.,requires_grad=True)
(3*w2-1).pow(2).backward();(3*w2-1).pow(2).backward()
print('Gradient accumulé sans zero_grad :',w2.grad.item())

## Les entrées, les logits et la perte

Le prétraitement est ajusté uniquement sur les lignes d’apprentissage de chaque fit. CrossEntropyLoss reçoit des logits et des classes entières.

Avant calcul : prévoir une sortie. Après calcul : nommer le résultat, sa limite et la décision.

In [ ]:
def prepare(train_x):
 # Prétraitement du modèle linéaire J2, réutilisé en dense pour le réseau.
 return clone(make_model('fraud',tr,'linear')[:-1])
def dense(a):
 return np.asarray(a.toarray() if hasattr(a,'toarray') else a,dtype=np.float32)
logits=torch.tensor([[0.,2.],[2.,0.]])
probas=torch.softmax(logits,dim=1)
print('Logits :',logits.tolist(),'probabilités classe 1 :',probas[:,1].tolist())
print('Cross-entropy :',nn.CrossEntropyLoss()(logits,torch.tensor([1,0])).item())
print('Ne pas ajouter softmax avant CrossEntropyLoss.')

## La boucle visible et les courbes

Architecture fixe : entrées → 32 neurones ReLU → 2 logits. Budget fixe de 8 epochs ; aucune sélection sur le contrôle externe.

Avant calcul : prévoir une sortie. Après calcul : nommer le résultat, sa limite et la décision.

In [ ]:
EPOCHS=8;BATCH=256
class FraudNet(nn.Module):
 def __init__(self,n):
  super().__init__();self.layers=nn.Sequential(nn.Linear(n,32),nn.ReLU(),nn.Linear(32,2))
 def forward(self,x):return self.layers(x)
def train_net(train_x,train_y,lr=.001,eval_x=None,eval_y=None):
 torch.manual_seed(2026)
 prep=prepare(train_x);a=torch.tensor(dense(prep.fit_transform(train_x)))
 target=torch.tensor(np.asarray(train_y),dtype=torch.long)
 net=FraudNet(a.shape[1]);opt=torch.optim.AdamW(net.parameters(),lr=lr,weight_decay=.01)
 criterion=nn.CrossEntropyLoss();history=[]
 g=torch.Generator().manual_seed(2026)
 b=torch.tensor(dense(prep.transform(eval_x))) if eval_x is not None else None
 for epoch in range(1,EPOCHS+1):
  net.train()
  for idx in torch.randperm(len(a),generator=g).split(BATCH):
   opt.zero_grad();logits=net(a[idx]);loss=criterion(logits,target[idx])
   loss.backward();opt.step()
  net.eval()
  with torch.no_grad():
   train_loss=criterion(net(a),target).item()
   row={'epoch':epoch,'train_loss':train_loss}
   if b is not None:
    out=net(b);row['valid_loss']=criterion(out,torch.tensor(np.asarray(eval_y),dtype=torch.long)).item()
    row['valid_auc']=roc_auc_score(eval_y,torch.softmax(out,1)[:,1].numpy())
  history.append(row)
 return prep,net,pd.DataFrame(history)
def predict_net(prep,net,frame):
 net.eval();parts=[]
 with torch.no_grad():
  for start in range(0,len(frame),4096):
   a=torch.tensor(dense(prep.transform(frame.iloc[start:start+4096])))
   parts.append(torch.softmax(net(a),dim=1)[:,1].numpy())
 return np.concatenate(parts)
# Premier fold seulement : une courbe de démonstration, pas le contrôle externe.
a,b=folds[0]
prep_demo,net_demo,history=train_net(X.iloc[a],y.iloc[a],eval_x=X.iloc[b],eval_y=y.iloc[b])
display(history.round(5));history.plot(x='epoch',y=['train_loss','valid_loss']);plt.show()
print('Paramètres du réseau :',sum(p.numel() for p in net_demo.parameters()))

## Deux tailles de pas, les mêmes folds

Deux learning rates × trois folds = six entraînements neuronaux de huit epochs. La démonstration précédente ajoute un entraînement.

Avant calcul : prévoir une sortie. Après calcul : nommer le résultat, sa limite et la décision.

In [ ]:
network_rows=[]
for lr in [.001,.01]:
 scores=[];start=time.perf_counter()
 for a,b in folds:
  pp,net,h=train_net(X.iloc[a],y.iloc[a],lr=lr)
  scores.append(roc_auc_score(y.iloc[b],predict_net(pp,net,X.iloc[b])))
 network_rows.append({'modele':'reseau','lr':lr,'auc_cv':np.mean(scores),'ecart_type':np.std(scores),'secondes':time.perf_counter()-start})
network_table=pd.DataFrame(network_rows)
base=make_model('fraud',tr,'linear',params={'C':.1})
start=time.perf_counter();res=cross_validate(base,X,y,cv=folds,scoring='roc_auc',n_jobs=1)
base_row={'modele':'logistique_C01','lr':np.nan,'auc_cv':res['test_score'].mean(),'ecart_type':res['test_score'].std(),'secondes':time.perf_counter()-start}
comparison=pd.concat([network_table,pd.DataFrame([base_row])],ignore_index=True)
display(comparison.round(5));comparison.to_csv(RUN/'comparaison.csv',index=False)
best_lr=float(network_table.sort_values('auc_cv',ascending=False).iloc[0].lr)
print('Meilleur learning rate du réseau :',best_lr,'; le réseau ne gagne pas nécessairement.')

## Le contrôle après le choix

Le contrôle local est connu depuis J2 ; on l’utilise pour décrire, sans prétendre disposer d’un nouveau test indépendant.

Avant calcul : prévoir une sortie. Après calcul : nommer le résultat, sa limite et la décision.

In [ ]:
prep_selected,net_selected,history_selected=train_net(X,y,lr=best_lr)
pred_valid=predict_net(prep_selected,net_selected,Xv)
control_auc=roc_auc_score(yv,pred_valid)
print('Réseau retenu parmi les deux learning rates : AUC contrôle =',round(control_auc,6))
print('Perte et AUC répondent à deux questions différentes ; aucune calibration démontrée.')
prior=ROOT/'runs/J3/tournoi.csv'
if prior.exists():
 print('Comparaison J3 (vérifier que le manifeste et le sous-échantillon correspondent) :');display(pd.read_csv(prior))
else:print('Tableau J3 absent : utiliser les résultats projetés et vérifier le protocole avant comparaison.')

## Un export neuronal pour Kaggle

Le CSV constitue un candidat. Le choix de l’envoyer dépend de la CV, du coût et des règles, pas de la nouveauté du modèle.

Avant calcul : prévoir une sortie. Après calcul : nommer le résultat, sa limite et la décision.

In [ ]:
from sklearn.model_selection import train_test_split
from mllab.competition import read_competition
all_train,test,sample=read_competition('fraud',root=ROOT)
fit_train,_=train_test_split(all_train,train_size=LIMIT,stratify=all_train.is_fraud,random_state=2026)
prep_final,net_final,history_final=train_net(xy(fit_train,'fraud'),fit_train.is_fraud.astype(int),lr=best_lr)
pred_test=predict_net(prep_final,net_final,xy(test,'fraud'))
submission=sample.copy();submission['is_fraud']=pred_test
assert list(submission)==['id','is_fraud'] and np.array_equal(submission.id,test.id)
assert np.isfinite(pred_test).all() and ((pred_test>=0)&(pred_test<=1)).all()
submission.to_csv(RUN/'submission_J4.csv',index=False)
torch.save({'state_dict':net_final.state_dict(),'n_inputs':net_final.layers[0].in_features,'lr':best_lr,'epochs':EPOCHS},RUN/'poids_reseau.pt')
manifest={'train_ids_sha256':ids_sha,'donnees_sha256':protocol_record['data_sha256'],'epochs':EPOCHS,'batch':BATCH,'lr':best_lr,'torch':torch.__version__,'appareil':'cpu','controle_auc':control_auc,'lignes_submission':len(submission),'submission_sha256':hashlib.sha256((RUN/'submission_J4.csv').read_bytes()).hexdigest(),'score_public':None,'note':'Le checkpoint de poids nécessite aussi le prétraitement ajusté ; le CSV est autonome.'}
(RUN/'manifest.json').write_text(json.dumps(manifest,indent=2))
history.to_csv(RUN/'courbe_demo.csv',index=False)
display(submission.head());print(len(submission),'lignes exportées ; aucun dépôt automatique.')

## La décision et le bilan

Ne pas attribuer une amélioration à PyTorch sans comparaison. Écrire une prochaine expérience limitée, par exemple 16 neurones au lieu de 32.

Avant calcul : prévoir une sortie. Après calcul : nommer le résultat, sa limite et la décision.

In [ ]:
journal={'hypothese':'À compléter : effet attendu du learning rate','preuve':'À compléter : comparaison CV, dispersion et durée','decision':'À compléter : garder réseau ou référence, et pourquoi','prochaine_experience':'Un changement, mêmes folds, budget explicite','score_public_reel':None}
(RUN/'journal.json').write_text(json.dumps(journal,indent=2,ensure_ascii=False))
print('Boucle : zero_grad → forward → loss → backward → step.')
print('Fichiers :',', '.join(sorted(p.name for p in RUN.iterdir() if p.is_file())))

## Remise et explication

Conserver le tableau des résultats, le manifeste, le CSV de prédictions et le journal. Compléter le journal avec une hypothèse, une conclusion chiffrée et une prochaine expérience. Chaque membre explique un choix et une limite. Vérifier les règles et l’ouverture de la compétition avant un dépôt manuel ; le CSV reste utilisable pour un exercice de format si le dépôt est fermé.
